# Part 02 — Member 1: Adstock & Feature Engineering

**Goal:** Regenerate `data/processed/mmm_features.csv` from Part 01 cleaned data, with the same features and column order already consumed by Member 2 and Member 3.

- **Target (Y):** `sales_revenue_lkr` (sales revenue, not profit).
- **Media inputs (X):** monthly LKR spend across TV, Facebook/Instagram, Google Search, YouTube, Radio, and Outdoor.
- **Other inputs (X):** price, promotions, competitor pressure, economic sentiment, exchange rate, inflation, seasonal demand, public holidays, and cyclic calendar month.
- **Do not train the model here:** Model training belongs to `02_model_training.ipynb`.

**Cautions:** This is 36 months of **synthetic** observational data. Adstock decay rates below are illustrative assumptions, not estimated from experiments. Correlations and modelled response are **not proof of causal advertising lift or ROI**.

## 1. Load the cleaned data and source transformation code

The notebook locates the project root whether it is launched from the repository or from the notebook folder. Reuse `src/adstock.py` instead of copying different Adstock logic into this notebook.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data/processed/cleaned_data.csv").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find data/processed/cleaned_data.csv; open this project first.")
sys.path.insert(0, str(PROJECT_ROOT))
from src.adstock import CHANNELS, DECAY_RATES, apply_adstock, adstock_half_life

CLEANED_PATH = PROJECT_ROOT / "data/processed/cleaned_data.csv"
FEATURE_PATH = PROJECT_ROOT / "data/processed/mmm_features.csv"
GRAPH_PATH = PROJECT_ROOT / "outputs/mmm/adstock_analysis.png"

cleaned = pd.read_csv(CLEANED_PATH, parse_dates=["month"]).sort_values("month").reset_index(drop=True)
assert cleaned["month"].is_unique and cleaned["month"].is_monotonic_increasing
assert cleaned["month"].notna().all()
assert (cleaned[CHANNELS] >= 0).all().all(), "Ad spend must be nonnegative"
print(f"Cleaned dataset: {len(cleaned)} monthly observations, {len(cleaned.columns)} columns")
print(f"Period: {cleaned['month'].min().date()} to {cleaned['month'].max().date()}")
display(cleaned[["month", *CHANNELS, "sales_revenue_lkr"]].head())

## 2. Identify features and the target

**Target:** sales revenue (`sales_revenue_lkr`).

**Media inputs:** Six raw monthly advertising spend columns are first transformed by geometric Adstock. The model receives the **adstocked** versions rather than raw spend totals.

**External controls:** Prices and promotional discounts; competitor pressure, economic sentiment, exchange rate and inflation; seasonal demand and public holidays. Month-of-year is encoded with sine and cosine to avoid treating December → January as a large discontinuity.

**Excluded:** `units_sold`, `impressions`, `clicks`, and `conversions` (possible outcomes/mediators or leakage), `total_ad_spend_lkr` (sum of the six channels), and non-numeric metadata.

In [ ]:
TARGET = "sales_revenue_lkr"
ADSTOCK_COLUMNS = [name.replace("_spend_lkr", "_adstock_lkr") for name in CHANNELS]
EXTERNAL_CONTROLS = [
    "average_price_index", "promotion_discount_pct", "competitor_pressure_index_0_100",
    "economic_sentiment_index_0_100", "lkr_usd_exchange_rate",
    "inflation_proxy_index", "seasonal_demand_index", "public_holiday_count",
]
required = [*CHANNELS, *EXTERNAL_CONTROLS, TARGET]
missing = [c for c in required if c not in cleaned.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")
if cleaned[required].isna().any().any():
    raise ValueError("Required modelling inputs contain missing values")
print("Target:", TARGET)
print("Media inputs:", CHANNELS)
print("External factors:", EXTERNAL_CONTROLS)

## 3. Geometric Adstock (advertising carryover)

Monthly formula for each channel:

\[ A_t = S_t + \lambda A_{t-1} \]

where `S_t` is this month's raw spend and `λ` is a channel-specific assumed monthly carryover rate. For the first historical observation, prior Adstock is assumed to be zero. This **initial-state assumption** matters for early months. For future one-month budget scenarios, `src/predict_sales.py` requires the previous month's **actual Adstock state** explicitly.

These decay values were chosen as illustrative assumptions, **not** estimated from the 36 observations. They should be treated cautiously during optimisation.

In [ ]:
decay_summary = pd.DataFrame({
    "spend_column": CHANNELS,
    "assumed_decay_rate": [DECAY_RATES[c] for c in CHANNELS],
    "approx_half_life_months": [adstock_half_life(DECAY_RATES[c]) for c in CHANNELS],
})
display(decay_summary)
transformed = apply_adstock(cleaned, decay_rates=DECAY_RATES)
assert (transformed[ADSTOCK_COLUMNS] >= 0).all().all()
display(transformed[["month", *ADSTOCK_COLUMNS]].head())

## 4. Add external and cyclic seasonal features

Keep the external controls unchanged from the cleaned dataset. Add month sine/cosine using `2π × calendar_month / 12`. This matches the existing Member 2 training input format. The target remains unchanged.

In [ ]:
calendar_month = transformed["month"].dt.month
transformed["month_sin"] = np.sin(2 * np.pi * calendar_month / 12)
transformed["month_cos"] = np.cos(2 * np.pi * calendar_month / 12)

output_columns = ["month", *ADSTOCK_COLUMNS, *EXTERNAL_CONTROLS,
                  "month_sin", "month_cos", TARGET]
features = transformed.loc[:, output_columns].copy()
assert features.shape == (36, 18), f"Unexpected feature shape: {features.shape}"
assert not features.isna().any().any()
print("Final model features:", list(features.columns))
display(features.head(3))

## 5. Plot raw spend versus transformed Adstock

Each panel overlays raw monthly spend and Adstock for one advertising channel. This is a visualization of the **assumed transformation**, not evidence of additional sales caused by that channel.

In [ ]:
GRAPH_PATH.parent.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(3, 2, figsize=(14, 11), constrained_layout=True)
for ax, raw, adstock_col in zip(axes.ravel(), CHANNELS, ADSTOCK_COLUMNS):
    ax.plot(transformed["month"], transformed[raw] / 1e6, label="Raw monthly spend", linewidth=1.6)
    ax.plot(transformed["month"], transformed[adstock_col] / 1e6, label="Adstock", linewidth=1.6)
    ax.set_title(raw.replace("_spend_lkr", "").replace("_", " ").title())
    ax.set_ylabel("LKR millions")
    ax.grid(alpha=0.2)
    ax.tick_params(axis="x", labelrotation=25)
    ax.legend(loc="best", fontsize=8)
fig.suptitle("Illustrative geometric Adstock by advertising channel", fontsize=14)
fig.savefig(GRAPH_PATH, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", GRAPH_PATH.relative_to(PROJECT_ROOT))

## 6. Reproduce the model feature handoff

Member 2 and Member 3 already use `data/processed/mmm_features.csv`. Before replacing that file, compare the new features with the existing one. They must match, so earlier model results are not invalidated. The file is only written after validation.

In [ ]:
if FEATURE_PATH.exists():
    existing = pd.read_csv(FEATURE_PATH, parse_dates=["month"])
    pd.testing.assert_frame_equal(
        features, existing, check_dtype=False, check_exact=False, atol=1e-7, rtol=1e-10
    )
    print("PASS: Reconstructed features match the previously submitted mmm_features.csv")
else:
    print("No existing feature CSV was found; a new file will be created.")

if not FEATURE_PATH.exists():
    features.to_csv(FEATURE_PATH, index=False, date_format="%Y-%m-%d")
    print("Created:", FEATURE_PATH.relative_to(PROJECT_ROOT))
else:
    print("Validated existing:", FEATURE_PATH.relative_to(PROJECT_ROOT))
reloaded = pd.read_csv(FEATURE_PATH)
assert reloaded.shape == (36, 18)
print("Feature engineering completed successfully.")

## 7. Summary, handover, and limitations

- The cleaned data were sorted chronologically, checked for missing values, and transformed without changing original source records.
- Six geometric Adstock features were generated with assumed channel-specific decay rates.
- Eight external controls plus two cyclical calendar-month features were added.
- Final handoff: **36 monthly rows and 18 columns**, including month and the sales target.
- Saved files: `data/processed/mmm_features.csv` and `outputs/mmm/adstock_analysis.png`.
- **No causal conclusions:** 36 synthetic observations cannot establish marketing incrementality, profit ROI, or real-world impact. Sales prediction is not profit prediction.